# Stage 05 — SCD2 & Star Schema

**Goal:** Explain as-of joins in a star schema and detect SCD2 history overlaps and duplicated facts.

**Setup:** Python 3.11+, Jupyter/VS Code; local synthetic examples only.

**CS bridge:** Explain the invariant and its failure mode before running code.

**Source:** https://www.postgresql.org/docs/current/queries-table-expressions.html

**Next:** Complete the graded lab under ../labs/ and pass an unseen transfer challenge.

## 1. Predict before running

Before selecting Run All, write down what the next assertion should prove and what it does **not** verify. Run the cell only after making your prediction.

In [ ]:
import sqlite3
db=sqlite3.connect(":memory:")
db.executescript("""
create table dim_region(code text, name text, valid_from text, valid_to text);
create table facts(event_id text, code text, observed_at text, magnitude real);
insert into dim_region values('E','Old East','2025-01-01','2026-01-02');
insert into dim_region values('E','New East','2026-01-02',null);
insert into facts values('A','E','2026-01-01',4.1);
insert into facts values('B','E','2026-01-02',5.7);
""")
result=db.execute("""
select f.event_id,d.name from facts f join dim_region d on f.code=d.code
where f.observed_at>=d.valid_from and
      (d.valid_to is null or f.observed_at<d.valid_to)
order by f.event_id
""").fetchall()
assert result==[("A","Old East"),("B","New East")]
print(result)

## 2. Independent failure check

On your own practice branch, change one parameter. Explain why the assertion fails or the output changes, then restore the original value. Preserve the invariant.

In [ ]:
incorrect=db.execute("""
select f.event_id,d.name from facts f join dim_region d on f.code=d.code
""").fetchall()
assert len(incorrect)==4 and len(result)==2
print("Counterexample: the wrong join doubles rows; index choice cannot fix wrong grain.")
db.close()

## 3. Unseen transfer (no AI)

Add a region with three historical versions and one late-arriving event. Prove that the event is not mistakenly joined to the latest version.

**Do not mark the stage complete based on this notebook alone.** Finish the corresponding graded lab and complete a mentor-reviewed transfer task.